In [ ]:
# input
high_conf_pred = "../predict_afdb/data/pred_ge_3_clique_3.tsv"
domain_assign = "./tmp/target_domain.tsv"
# output
sites_domain_file = "./tmp/entryId-tedIds-posis.tsv"

In [2]:
import pandas as pd

df_pred = pd.read_table(high_conf_pred)
df_pred['seq_id'] = df_pred['seq_id'].map(lambda x: x.split("-")[1])
seq_id_to_posi = dict(zip(df_pred["seq_id"], df_pred['posi']))
del df_pred

df_domain = pd.read_table(domain_assign, header=None, names=["seq_id", "redundant_id", "entry_id", "rep_id", "domain"], na_values=[], keep_default_na=False)

In [3]:
def get_all_positions_to_domain_id(domain_id_to_ranges_str: dict):
    result = [None for _ in range(2700)]
    for k, v in domain_id_to_ranges_str.items():
        for range_str in v.split("_"):
            start, stop = range_str.split("-")
            start, stop = int(start), int(stop)
            length = stop - start + 1
            result[(start - 1): stop] = [k for _ in range(length)]

    return result

In [4]:
import tqdm
import csv

with open(sites_domain_file, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t", lineterminator="\n")

    for (seq_id,), df_seq_id in tqdm.tqdm(df_domain.groupby(by=['seq_id'])):
        all_positions_to_domain_id = get_all_positions_to_domain_id(dict(zip(df_seq_id['entry_id'], df_seq_id['domain'])))

        pred_positions = seq_id_to_posi[seq_id]
        domain2positions = dict()
        for p in pred_positions.split(","):
            d = all_positions_to_domain_id[int(p)]
            if d is not None:
                if d not in domain2positions:
                    domain2positions[d] = [p]
                else:
                    domain2positions[d].append(p)
    
        domains = []
        positions = []
        for k, v in domain2positions.items():
            domains.append(k)
            positions.append(",".join(v))

        if len(domains) > 0:
            _ = writer.writerow([seq_id, ",".join(domains), ";".join(positions)])

100%|██████████| 37623404/37623404 [1:21:03<00:00, 7736.27it/s]
